<a href="https://colab.research.google.com/github/sumbul-ahmad/rag-document-qa/blob/main/rag_document_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers anthropic numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.4 MB/s eta 0:00:00


In [1]:
import torch
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device=0,
)
print("Model loaded")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Your documents (we'll replace these with your own lecture notes later)
docs = [
    "Gradient descent updates model weights in the direction that reduces the loss.",
    "Overfitting happens when a model memorizes training data and fails on new data.",
    "Regularization techniques like dropout and weight decay reduce overfitting.",
    "A learning rate that is too high can make training unstable and diverge.",
    "Cross-validation splits data into folds to estimate how well a model generalizes.",
]

# 2. Turn each document into a vector (embedding)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_vecs = embedder.encode(docs, normalize_embeddings=True)

# 3. Find the most relevant documents for a question
def retrieve(question, k=2):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = doc_vecs @ q
    top = np.argsort(scores)[::-1][:k]
    return [docs[i] for i in top]

print(retrieve("How can I stop my model from overfitting?"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

['Overfitting happens when a model memorizes training data and fails on new data.', 'Regularization techniques like dropout and weight decay reduce overfitting.']


In [3]:
def answer(question):
    context = "\n".join(retrieve(question))
    messages = [
        {"role": "system", "content": "Answer the question using only the given context. If the context does not contain the answer, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    out = generator(messages, max_new_tokens=150, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

print(answer("How can I stop my model from overfitting?"))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


You can use regularization techniques such as dropout or weight decay to prevent your model from overfitting.


In [4]:
print(answer("What is the capital of France?"))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I don't know. The context provided does not contain any information about the capital of France or machine learning concepts like cross-validation or learning rates.
